In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes

DATA_DIR = Path.cwd().parent / "data"
df = add_regimes(add_features(load_processed("BTC/USDT", DATA_DIR)))

from cryptosense.setups import explain_setup
from cryptosense.setups import setup_flag


## Setup definition (written before computing any outcomes)

**Pullback in an uptrend:** flagged when `dist_sma_100 > 0` (long-term trend up) and `dist_sma_20 < 0` (price below its 20-day average). No tuned thresholds. Long-only. Development period: data up to 2023-12-31; 2024 onward is held out for a single final check.

In [2]:
dev = df.loc[:"2023-12-31"].dropna(subset=["dist_sma_100", "dist_sma_20"])
dev["setup_pullback"] = (dev["dist_sma_100"] > 0) & (dev["dist_sma_20"] < 0)

print("days in development period:", len(dev))
print("share of days flagged:", round(dev["setup_pullback"].mean(), 3))

days in development period: 2229
share of days flagged: 0.155


In [3]:
run_id = (dev["setup_pullback"] != dev["setup_pullback"].shift()).cumsum().rename("run")
runs = dev.groupby(run_id)["setup_pullback"].agg(["first", "size"])
runs.columns = ["flagged", "days"]
episodes = runs[runs["flagged"]]

print("episodes:", len(episodes))
print("median episode length (days):", episodes["days"].median())
episodes["days"].describe()

episodes: 78
median episode length (days): 2.0


count    78.000000
mean      4.435897
std       4.968968
min       1.000000
25%       1.000000
50%       2.000000
75%       6.750000
max      25.000000
Name: days, dtype: float64

In [4]:
HORIZONS = [5, 10]
for h in HORIZONS:
    df[f"fwd_ret_{h}"] = np.log(df["close"].shift(-h) / df["close"])

dev = df.loc[:"2023-12-31"].iloc[:-max(HORIZONS)]
dev = dev.dropna(subset=["dist_sma_100", "dist_sma_20"])
dev["setup_pullback"] = (dev["dist_sma_100"] > 0) & (dev["dist_sma_20"] < 0)

starts = dev["setup_pullback"] & ~dev["setup_pullback"].shift(fill_value=False)
entries = dev[starts]

def summarise(frame, label):
    row = {"group": label, "n": len(frame)}
    for h in HORIZONS:
        r = frame[f"fwd_ret_{h}"]
        row[f"mean_{h}d"] = r.mean()
        row[f"median_{h}d"] = r.median()
        row[f"positive_{h}d"] = (r > 0).mean()
    return row

comparison = pd.DataFrame(
    [summarise(dev, "all days"), summarise(entries, "pullback entries")]
).set_index("group")

print("dev ends:", dev.index[-1].date(), "| episodes:", len(entries))
comparison.round(4)

dev ends: 2023-12-21 | episodes: 75


,n,mean_5d,median_5d,positive_5d,mean_10d,median_10d,positive_10d
group,,,,,,,
all days,2219,0.0035,0.0049,0.5300,0.0067,0.0072,0.5376
pullback entries,75,-0.0084,-0.0057,0.4667,-0.0057,-0.0025,0.4933


In [5]:
for h in HORIZONS:
    r = entries[f"fwd_ret_{h}"]
    se = r.std() / np.sqrt(len(r))
    diff = r.mean() - dev[f"fwd_ret_{h}"].mean()
    print(f"{h}d: difference = {diff:.4f} | SE = {se:.4f} | difference / SE = {diff / se:.2f}")

5d: difference = -0.0119 | SE = 0.0091 | difference / SE = -1.32
10d: difference = -0.0124 | SE = 0.0131 | difference / SE = -0.95


### Result: pullback in an uptrend (development period, to 2023-12-21)

**Observation:** 75 episodes (15.5% of days flagged, median episode length 2 days). After episode entries, the mean 5-day log return was −0.84% versus +0.35% for all days (difference −1.32 standard errors), and the mean 10-day return was −0.57% versus +0.67% (−0.95 standard errors). 46.7% of entries were followed by a positive 5-day return, versus 53.0% of all days.

**Interpretation:** No detectable edge. The slight underperformance is within what chance could produce, and overlapping outcomes make the true uncertainty somewhat larger than the standard errors suggest. One possible explanation, not tested here, is short-term momentum: drops below the 20-day average may tend to continue briefly rather than reverse. The rule is not being modified in response to this result.

In [6]:
dev["setup_continuation"] = (
    (dev["mom_20"] > 0)
    & (dev["er_20"] > 0.4)
    & (dev["rel_volume_20"] > 1.5)
)

cont_starts = dev["setup_continuation"] & ~dev["setup_continuation"].shift(fill_value=False)
print("share of days flagged:", round(dev["setup_continuation"].mean(), 3))
print("episodes:", int(cont_starts.sum()))

share of days flagged: 0.027
episodes: 34


In [7]:
cont_entries = dev[cont_starts]

comparison.loc["continuation entries"] = pd.Series(summarise(cont_entries, "continuation entries")).drop("group")

for h in HORIZONS:
    r = cont_entries[f"fwd_ret_{h}"]
    se = r.std() / np.sqrt(len(r))
    diff = r.mean() - dev[f"fwd_ret_{h}"].mean()
    print(f"{h}d: difference = {diff:.4f} | SE = {se:.4f} | difference / SE = {diff / se:.2f}")

comparison.round(4)

5d: difference = 0.0282 | SE = 0.0139 | difference / SE = 2.02
10d: difference = 0.0482 | SE = 0.0232 | difference / SE = 2.08


,n,mean_5d,median_5d,positive_5d,mean_10d,median_10d,positive_10d
group,,,,,,,
all days,2219,0.0035,0.0049,0.5300,0.0067,0.0072,0.5376
pullback entries,75,-0.0084,-0.0057,0.4667,-0.0057,-0.0025,0.4933
continuation entries,34,0.0317,0.0384,0.7647,0.0549,0.0314,0.6471


In [8]:
by_year = cont_entries.groupby(cont_entries.index.year).agg(
    episodes=("fwd_ret_5", "size"),
    mean_5d=("fwd_ret_5", "mean"),
    positive_5d=("fwd_ret_5", lambda r: (r > 0).mean()),
)
by_year.round(4)

,episodes,mean_5d,positive_5d
date,,,
2017,3,0.1150,1.0000
2018,2,0.0112,0.5000
2019,6,0.0092,0.8333
2020,13,0.0305,0.6923
2021,3,-0.0251,0.6667
2023,7,0.0475,0.8571


In [9]:
year_base = dev.groupby(dev.index.year)["fwd_ret_5"].mean().rename("base_5d")

within = by_year.join(year_base)
within["excess_5d"] = within["mean_5d"] - within["base_5d"]
within.round(4)

,episodes,mean_5d,positive_5d,base_5d,excess_5d
date,,,,,
2017,3,0.1150,1.0000,0.0657,0.0493
2018,2,0.0112,0.5000,-0.0188,0.0300
2019,6,0.0092,0.8333,0.0088,0.0004
2020,13,0.0305,0.6923,0.0203,0.0102
2021,3,-0.0251,0.6667,0.0050,-0.0301
2023,7,0.0475,0.8571,0.0134,0.0341


### Result: volume-confirmed continuation (development period, to 2023-12-21)

**Observation:** 34 episodes (2.7% of days flagged). After episode entries, the mean 5-day log return was +3.17% versus +0.35% for all days (2.02 standard errors), and the mean 10-day return was +5.49% versus +0.67% (2.08 standard errors). 76.5% of entries were followed by a positive 5-day return, versus 53.0% of all days; the median 5-day return was +3.84%. Episodes were concentrated in strong years (13 in 2020, none in 2022). Compared with an average day in the same year, the excess 5-day return was positive in 5 of 6 years, but the episode-weighted average excess was about +1.4% (roughly 1 standard error). The years with the most episodes, 2019 and 2020, showed the smallest excess (+0.0% and +1.0%).

**Interpretation:** About half of the initial advantage reflected bull-market timing: the setup fires mainly in strong years, which have high returns anyway. The remaining within-year advantage is small and not statistically established. Four comparisons were made (two setups at two horizons), so one borderline result is weaker evidence than it appears in isolation. Trading costs are not yet included. The rule was not modified in response to these results.

### Predictions for the held-out test (written before running it)

- Pullback: [beats / doesn't beat] the 2024+ base rate; difference [above / below] 2 standard errors.
- Continuation: [beats / doesn't beat] the 2024+ base rate; difference [above / below] 2 standard errors.

In [10]:
# 1. Flags and episode starts on the FULL history, so episodes crossing into 2024 aren't miscounted
df["setup_pullback"] = (df["dist_sma_100"] > 0) & (df["dist_sma_20"] < 0)
df["setup_continuation"] = (
    (df["mom_20"] > 0)
    & (df["er_20"] > 0.4)
    & (df["rel_volume_20"] > 1.5)
)
for name in ["pullback", "continuation"]:
    flag = df[f"setup_{name}"]
    df[f"{name}_start"] = flag & ~flag.shift(fill_value=False)

# 2. The held-out period, keeping only days with known outcomes
test = df.loc["2024-01-01":].dropna(subset=["fwd_ret_5", "fwd_ret_10"])
print("test:", test.index[0].date(), "->", test.index[-1].date(), "|", len(test), "days")

# 3. Each setup against the test period's own base rate
rows = [summarise(test, "all test days")]
for name in ["pullback", "continuation"]:
    entries_t = test[test[f"{name}_start"]]
    rows.append(summarise(entries_t, f"{name} entries"))
    for h in HORIZONS:
        r = entries_t[f"fwd_ret_{h}"]
        if len(r) < 2:
            print(f"{name} {h}d: too few episodes ({len(r)})")
            continue
        se = r.std() / np.sqrt(len(r))
        diff = r.mean() - test[f"fwd_ret_{h}"].mean()
        print(f"{name} {h}d: difference = {diff:.4f} | SE = {se:.4f} | difference / SE = {diff / se:.2f}")

test_comparison = pd.DataFrame(rows).set_index("group")
print(test_comparison.round(4))

# 4. Continuation by year, compared with each test year's own average day
cont_test = test[test["continuation_start"]]
by_year_test = cont_test.groupby(cont_test.index.year).agg(
    episodes=("fwd_ret_5", "size"),
    mean_5d=("fwd_ret_5", "mean"),
    positive_5d=("fwd_ret_5", lambda r: (r > 0).mean()),
).join(test.groupby(test.index.year)["fwd_ret_5"].mean().rename("base_5d"))
by_year_test["excess_5d"] = by_year_test["mean_5d"] - by_year_test["base_5d"]
by_year_test.round(4)

test: 2024-01-01 -> 2026-10-06 | 1010 days
pullback 5d: difference = -0.0015 | SE = 0.0103 | difference / SE = -0.15
pullback 10d: difference = 0.0029 | SE = 0.0114 | difference / SE = 0.25
continuation 5d: difference = 0.0182 | SE = 0.0119 | difference / SE = 1.52
continuation 10d: difference = 0.0325 | SE = 0.0154 | difference / SE = 2.10
                         n  mean_5d  median_5d  positive_5d  mean_10d  \
group                                                                   
all test days         1010   0.0031     0.0018       0.5208    0.0061   
pullback entries        33   0.0016    -0.0088       0.3333    0.0090   
continuation entries    17   0.0213     0.0081       0.6471    0.0385   

                      median_10d  positive_10d  
group                                           
all test days             0.0028        0.5267  
pullback entries         -0.0044        0.4545  
continuation entries      0.0284        0.8235  


,episodes,mean_5d,positive_5d,base_5d,excess_5d
date,,,,,
2024,7,0.0393,0.7143,0.0108,0.0285
2025,6,0.0047,0.6667,-0.0009,0.0056
2026,4,0.0147,0.5000,-0.0017,0.0164


### Held-out test results (2024-01-01 to 2026-10-06, run once)

Both rules were applied exactly as pre-registered, with episode starts computed on the full history and each setup compared with all days in the test period.

**Observation — pullback in an uptrend:** 33 episodes. Mean 5-day log return +0.16% versus +0.31% for all test days (−0.15 standard errors); mean 10-day return +0.90% versus +0.61% (+0.25 standard errors). 33.3% of entries were followed by a positive 5-day return, versus 52.1% of all days.

**Observation — volume-confirmed continuation:** 17 episodes. Mean 5-day log return +2.13% versus +0.31% (+1.52 standard errors); mean 10-day return +3.85% versus +0.61% (+2.10 standard errors). 82.4% of entries were followed by a positive 10-day return, versus 52.7% of all days. Compared with an average day in the same year, the 5-day excess was positive in every test year: +2.9% in 2024 (7 episodes), +0.6% in 2025 (6) and +1.6% in 2026 (4).

**Interpretation:** The pullback setup showed no edge in either the development or the held-out period. The continuation setup's modest advantage replicated on data it had never been tested on: about +1.4% per 5 days over an average day in the same period during development, and about +1.8% in the test period. Its positive excess in every test year means bull-market timing cannot explain the test result. The evidence nonetheless rests on few episodes (34 in development, 17 in test), with each individual comparison at roughly 1.5–2 standard errors; its strength comes from consistency across two independent periods rather than from any single number.

**Conclusion:** The continuation setup is a credible candidate for Phase 6 backtesting, where trading costs, slippage and explicit exit rules will determine whether the advantage survives in practice. It is not an established profitable strategy. Neither rule was modified at any point in response to results.

In [11]:
cont_flags = setup_flag(df, "continuation")
latest = cont_flags[cont_flags].index[-1]

import pprint
pprint.pprint(explain_setup(df, latest, "continuation"), sort_dicts=False)

{'date': datetime.date(2026, 8, 25),
 'setup': 'continuation',
 'flagged': True,
 'conditions': [{'feature': 'mom_20',
                 'rule': '> 0.0',
                 'value': 0.1944,
                 'met': True},
                {'feature': 'er_20',
                 'rule': '> 0.4',
                 'value': 0.6166,
                 'met': True},
                {'feature': 'rel_volume_20',
                 'rule': '> 1.5',
                 'value': 1.548,
                 'met': True}],
 'trend': 'up',
 'vol_regime': 'high',
 'daily_vol_forecast': 0.0216,
 'typical_move': 0.0483,
 'past_episodes': 54,
 'past_mean_return': 0.0255,
 'past_share_positive': 0.722,
 'past_mean_se': 0.0099,
 'base_mean_return': 0.0045,
 'base_same_trend_mean': 0.0115}
